In [ ]:
from pathlib import Path
import pandas as pd
import json

import sys
sys.path.append("../../../utils/")

from utils import *

In [ ]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[3]

NOMBRE_DATASET_ENTRADA = "BCCC17__cleanning__v1"
NOMBRE_SPLIT = "BCCC17__split__v1"

RUTA_DATASET_LIMPIO = PROJECT_ROOT / "02_datasets" / "processed_analisis_estadistico" / NOMBRE_DATASET_ENTRADA
RUTA_SALIDA = PROJECT_ROOT / "02_datasets" / "processed_analisis_estadistico" / NOMBRE_SPLIT

NOMBRE_DATASET_LIMPIO = f"{NOMBRE_DATASET_ENTRADA}.csv"
NOMBRE_TRAIN = f"{NOMBRE_SPLIT}__train.csv"
NOMBRE_TEST = f"{NOMBRE_SPLIT}__test.csv"
NOMBRE_REPORTE = f"{NOMBRE_SPLIT}_report.json"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"
TEST_SIZE = 0.20
RANDOM_STATE = 42

In [ ]:
print("PROJECT_ROOT:")
print(PROJECT_ROOT)
print()

print("Dataset limpio de entrada:")
print(RUTA_DATASET_LIMPIO / NOMBRE_DATASET_LIMPIO)
print()

print("Ruta de salida:")
print(RUTA_SALIDA)

In [ ]:
input_path = RUTA_DATASET_LIMPIO / NOMBRE_DATASET_LIMPIO

if not input_path.exists():
    raise FileNotFoundError(f"No existe el dataset limpio en: {input_path}")

df = cargar_dataset(nombre_dataset=NOMBRE_DATASET_LIMPIO, ruta_base=RUTA_DATASET_LIMPIO)

shape_original = df.shape

print("Forma del dataset limpio:")
print(shape_original)

In [ ]:
df.head()

In [ ]:
if LABEL_COL not in df.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL}")

print("Columna objetivo encontrada correctamente.")
print()
print("Distribución global de clases:")
display(resumen_clases(df, LABEL_COL))

In [ ]:
train_df, test_df = dividir_train_test_stratified(
    df=df,
    label_col=LABEL_COL,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("Forma train:", train_df.shape)
print("Forma test:", test_df.shape)

In [ ]:
print("Distribución de clases en TRAIN:")
display(resumen_clases(train_df, LABEL_COL))

In [ ]:
print("Distribución de clases en TEST:")
display(resumen_clases(test_df, LABEL_COL))

In [ ]:
resumen_global = resumen_clases(df, LABEL_COL).rename(
    columns={"count": "global_count", "percentage": "global_percentage"}
)

resumen_train = resumen_clases(train_df, LABEL_COL).rename(
    columns={"count": "train_count", "percentage": "train_percentage"}
)

resumen_test = resumen_clases(test_df, LABEL_COL).rename(
    columns={"count": "test_count", "percentage": "test_percentage"}
)

comparacion = pd.concat([resumen_global, resumen_train, resumen_test], axis=1)

print("Comparación global / train / test:")
display(comparacion)

In [ ]:
guardar_dataset_csv(
    df=train_df,
    nombre_archivo=NOMBRE_TRAIN,
    ruta=RUTA_SALIDA
)

guardar_dataset_csv(
    df=test_df,
    nombre_archivo=NOMBRE_TEST,
    ruta=RUTA_SALIDA
)

print("Train guardado en:")
print(RUTA_SALIDA / NOMBRE_TRAIN)
print()
print("Test guardado en:")
print(RUTA_SALIDA / NOMBRE_TEST)

In [ ]:
reporte_split = {
    "dataset_entrada": NOMBRE_DATASET_LIMPIO,
    "label_column": LABEL_COL,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "shape_global": {
        "rows": int(df.shape[0]),
        "cols": int(df.shape[1])
    },
    "shape_train": {
        "rows": int(train_df.shape[0]),
        "cols": int(train_df.shape[1])
    },
    "shape_test": {
        "rows": int(test_df.shape[0]),
        "cols": int(test_df.shape[1])
    },
    "class_distribution_global": {
        str(k): int(v) for k, v in df[LABEL_COL].value_counts(dropna=False).to_dict().items()
    },
    "class_distribution_train": {
        str(k): int(v) for k, v in train_df[LABEL_COL].value_counts(dropna=False).to_dict().items()
    },
    "class_distribution_test": {
        str(k): int(v) for k, v in test_df[LABEL_COL].value_counts(dropna=False).to_dict().items()
    }
}

report_path = RUTA_SALIDA / NOMBRE_REPORTE

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(reporte_split, f, indent=2, ensure_ascii=False)

print("Reporte guardado en:")
print(report_path)

In [ ]:
print("========== RESUMEN SPLIT ==========")
print(f"Dataset de entrada: {NOMBRE_DATASET_LIMPIO}")
print(f"Forma global: {df.shape}")
print(f"Forma train: {train_df.shape}")
print(f"Forma test: {test_df.shape}")
print(f"Test size: {TEST_SIZE}")
print(f"Random state: {RANDOM_STATE}")
print("===================================")